# Amazon US BSR Share 

In [3]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import re
import math
from datetime import date

In [4]:
import os
import sys

import io
from io import BytesIO
import csv

import google.auth
from google.cloud import bigquery
#from google.cloud import bigquery_storage

In [5]:
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "../market-analysis-project-91130-5213911f50a5.json"

credentials, project_id = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)

bqclient = bigquery.Client(credentials=credentials, project=project_id,)

In [9]:
sql = f"""
select * from wook.amz_us_bsr_add_seller_price_rating
where bsr_ctgry in ('Mattresses','Beds','Bed Frames') and rank <= 20 and bsr_date <= '2025-12-31'
"""

df_bsr = bqclient.query(sql).to_dataframe()

In [11]:
print(df_bsr)

       parent_asin   bsr_ctgry        asin  rank    bsr_date   brand  \
0       B084Q4Z5ZJ        Beds  B081HXZ5NK    17  2022-10-23  Others   
1       B0B38RKRB3  Mattresses  B0827DFY3B    19  2022-10-23  Others   
2       B0BSRPX3YF        Beds  B092LNV521    16  2022-10-23  Others   
3       B0CSZLZM7L  Mattresses  B07ZKMYFJZ    17  2022-10-23  Others   
4       B0DKFG8TSB  Bed Frames  B07C2VZ4G5    19  2022-10-23  Others   
...            ...         ...         ...   ...         ...     ...   
108393  B0DHRTBNV1  Bed Frames  B0CSYBPMW4     2  2024-11-12   ZINUS   
108394  B0F193LJW8  Mattresses  B0CSK2VDKG     6  2024-11-12   ZINUS   
108395  B0D8K4N7KM  Mattresses  B0CKYRS739    14  2024-11-12   ZINUS   
108396  B0DDXJLFWD        Beds  B071JGBZYW    17  2024-11-12   ZINUS   
108397  B0DYZ28XZV  Mattresses  B0CP1LR1PW     2  2024-11-12   ZINUS   

       brand_raw brand_raw_org  \
0         Others        TUSEER   
1         Others  S SECRETLAND   
2         Others     EDENBROOK   

In [113]:
# Stackline Sales
sql = f"""
select * from stck.atlas_sales_all 
where SubCategory in ('Mattresses','Beds','Bed Frames') 
    and WeekId between 202201 and 202552
"""

df_sales = bqclient.query(sql).to_dataframe()

In [115]:
print(df_sales)

         RetailerId RetailerName RetailerSku  upc          ModelNumber  \
0                 1   Amazon.com  B09PFVMWKC  NaN                 None   
1                 1   Amazon.com  B082V3R9YM  NaN                 None   
2                 1   Amazon.com  B07YQRL4G5  NaN                 King   
3                 1   Amazon.com  B07DC3JJXN  NaN            AR9016031   
4                 1   Amazon.com  B09BJ4WMMD  NaN             AM-STT-Q   
...             ...          ...         ...  ...                  ...   
2967631           1   Amazon.com  B0DQ885S4N  NaN  14 in king mattress   
2967632           1   Amazon.com  B0DDPQZQYB  NaN          SW-TFC4F-H2   
2967633           1   Amazon.com  B0876Z43S2  NaN           MSHEBT-13Q   
2967634           1   Amazon.com  B0CSFGYRHT  NaN           AOMS10QWP6   
2967635           1   Amazon.com  B0DHXMG94Z  NaN         DYO-14K-BLM2   

                                                     Title        Brand  \
0        14 Inch 4000lbs Heavy Duty 

In [19]:
df_bsr.duplicated().sum()

0

In [144]:
df1 = df_bsr.copy()

df1['bsr_date'] = pd.to_datetime(df1['bsr_date'], errors='coerce')
df1['year'] = df1['bsr_date'].dt.year
print(df1)

       parent_asin   bsr_ctgry        asin  rank   bsr_date   brand brand_raw  \
0       B084Q4Z5ZJ        Beds  B081HXZ5NK    17 2022-10-23  Others    Others   
1       B0B38RKRB3  Mattresses  B0827DFY3B    19 2022-10-23  Others    Others   
2       B0BSRPX3YF        Beds  B092LNV521    16 2022-10-23  Others    Others   
3       B0CSZLZM7L  Mattresses  B07ZKMYFJZ    17 2022-10-23  Others    Others   
4       B0DKFG8TSB  Bed Frames  B07C2VZ4G5    19 2022-10-23  Others    Others   
...            ...         ...         ...   ...        ...     ...       ...   
108393  B0DHRTBNV1  Bed Frames  B0CSYBPMW4     2 2024-11-12   ZINUS     ZINUS   
108394  B0F193LJW8  Mattresses  B0CSK2VDKG     6 2024-11-12   ZINUS     ZINUS   
108395  B0D8K4N7KM  Mattresses  B0CKYRS739    14 2024-11-12   ZINUS     ZINUS   
108396  B0DDXJLFWD        Beds  B071JGBZYW    17 2024-11-12   ZINUS     ZINUS   
108397  B0DYZ28XZV  Mattresses  B0CP1LR1PW     2 2024-11-12   ZINUS     ZINUS   

       brand_raw_org       

In [146]:
def get_brand_share_by_category(
    df,
    category,
    years=[2025],
    category_col='bsr_ctgry',
    brand_col='brand_raw_org',
    top_n=10,
    exclude_others=True
):
    """
    특정 category(bsr_ctgry) 기준으로
    - brand BSR 노출 수
    - category 내 brand share 계산
    - Top N 브랜드 반환
    """

    # === 1. 연도 + 카테고리 필터 ===
    df_use = df[
        (df['year'].isin(years)) &
        (df[category_col] == category)
    ].copy()

    if df_use.empty:
        print(f"[INFO] No data for category='{category}', years={years}")
        return pd.DataFrame()

    # === 2. (category, brand) BSR 노출 횟수 ===
    brand_cnt = (
        df_use
        .groupby(brand_col)
        .size()
        .reset_index(name='bsr_cnt')
    )

    # === 3. category 기준 share 계산 ===
    brand_share = (
        brand_cnt
        .assign(
            cat_total_cnt=lambda x: x['bsr_cnt'].sum(),
            bsr_share=lambda x: x['bsr_cnt'] / x['cat_total_cnt'],
            bsr_share_pct=lambda x: x['bsr_share'] * 100
        )
        .sort_values('bsr_share', ascending=False)
    )
    #print(brand_share)

    # === 5. Top N ===
    return brand_share.head(top_n)

In [148]:
categories = ['Beds', 'Bed Frames', 'Mattresses']

top10_base_2025 = {
    cat: get_brand_share_by_category(df1, category=cat, years=[2025])
    for cat in categories
}

In [150]:
top10_base_2025['Bed Frames']

,brand_raw_org,bsr_cnt,cat_total_cnt,bsr_share,bsr_share_pct
0,ALLEWIE,857,7300,0.117397,11.739726
1,AMAZON BASICS,733,7300,0.100411,10.041096
46,YAHEETECH,491,7300,0.067260,6.726027
48,ZINUS,490,7300,0.067123,6.712329
15,FURNULEM,431,7300,0.059041,5.904110
34,NEW JETO,365,7300,0.050000,5.000000
33,MELLOW,365,7300,0.050000,5.000000
27,LIKIMIO,365,7300,0.050000,5.000000
18,HLIPHA,365,7300,0.050000,5.000000
16,HAFENPO,365,7300,0.050000,5.000000


#### 1. 년도별 점유율 계산

In [152]:
# result_all 값을 기준으로 is_top10 컬럼만들기 

# Dict --> Data Frame으로 변경
rows = []
for cat, topdf in top10_base_2025.items():
    brands = topdf['brand_raw_org']
    #brands = brands.dropna().astype(str).unique().tolist()
    rows.extend([(cat, b) for b in brands])

#print(rows)

top_map = (
    pd.DataFrame(rows, columns=['bsr_ctgry', 'brand_raw_org'])
    .drop_duplicates()
)

top_map['is_top10'] = 1
print(top_map)

     bsr_ctgry        brand_raw_org  is_top10
0         Beds           SHA CERLIN         1
1         Beds               DICTAC         1
2         Beds             ADORNEVE         1
3         Beds              ALLEWIE         1
4         Beds             AMERLIFE         1
5         Beds               VINGLI         1
6         Beds               MELLOW         1
7         Beds                ZINUS         1
8         Beds             MILLIARD         1
9         Beds             JOCOEVOL         1
10  Bed Frames              ALLEWIE         1
11  Bed Frames        AMAZON BASICS         1
12  Bed Frames            YAHEETECH         1
13  Bed Frames                ZINUS         1
14  Bed Frames             FURNULEM         1
15  Bed Frames             NEW JETO         1
16  Bed Frames               MELLOW         1
17  Bed Frames              LIKIMIO         1
18  Bed Frames               HLIPHA         1
19  Bed Frames              HAFENPO         1
20  Mattresses                ZINU

In [154]:
df_w_top = df1.merge( top_map, on=['bsr_ctgry', 'brand_raw_org'], how='left')
df_w_top['brand_new'] = df_w_top['brand_raw_org'].where(df_w_top['is_top10']==1, 'OTHERS')
print(df_w_top)

       parent_asin   bsr_ctgry        asin  rank   bsr_date   brand brand_raw  \
0       B084Q4Z5ZJ        Beds  B081HXZ5NK    17 2022-10-23  Others    Others   
1       B0B38RKRB3  Mattresses  B0827DFY3B    19 2022-10-23  Others    Others   
2       B0BSRPX3YF        Beds  B092LNV521    16 2022-10-23  Others    Others   
3       B0CSZLZM7L  Mattresses  B07ZKMYFJZ    17 2022-10-23  Others    Others   
4       B0DKFG8TSB  Bed Frames  B07C2VZ4G5    19 2022-10-23  Others    Others   
...            ...         ...         ...   ...        ...     ...       ...   
108393  B0DHRTBNV1  Bed Frames  B0CSYBPMW4     2 2024-11-12   ZINUS     ZINUS   
108394  B0F193LJW8  Mattresses  B0CSK2VDKG     6 2024-11-12   ZINUS     ZINUS   
108395  B0D8K4N7KM  Mattresses  B0CKYRS739    14 2024-11-12   ZINUS     ZINUS   
108396  B0DDXJLFWD        Beds  B071JGBZYW    17 2024-11-12   ZINUS     ZINUS   
108397  B0DYZ28XZV  Mattresses  B0CP1LR1PW     2 2024-11-12   ZINUS     ZINUS   

       brand_raw_org       

In [49]:
# 월별 집계하기 
brand_cnt_m = (
    df_w_top
    #.groupby(['bsr_ctgry', 'yr_month', 'brand_new'])
    .groupby(['bsr_ctgry', 'year', 'brand_new'])
    .size()
    .reset_index(name='bsr_cnt')
)

print(brand_cnt_m)

      bsr_ctgry  year      brand_new  bsr_cnt
0    Bed Frames  2021        ALLEWIE       52
1    Bed Frames  2021  AMAZON BASICS     1231
2    Bed Frames  2021         OTHERS     2668
3    Bed Frames  2021      YAHEETECH       24
4    Bed Frames  2021          ZINUS     3104
..          ...   ...            ...      ...
118  Mattresses  2025        MOLBIUS      290
119  Mattresses  2025        NOVILLA     1012
120  Mattresses  2025         OTHERS     1947
121  Mattresses  2025      ROLANSTAR      209
122  Mattresses  2025          ZINUS     1688

[123 rows x 4 columns]


In [156]:
# (category, year_month)별 분모 계산 + share
brand_share_m = (
    brand_cnt_m
    .assign(
        year_total_cnt=lambda x: x.groupby(['bsr_ctgry', 'year'])['bsr_cnt'].transform('sum'),
        bsr_share=lambda x: x['bsr_cnt'] / x['year_total_cnt'],
       # bsr_share_pct=lambda x: x['bsr_share'] * 100
    )
    .sort_values(['bsr_ctgry', 'year', 'bsr_share'], ascending=[True, True, False])
)

#brand_share_m[brand_share_m['brand_new']=='OTHERS']
print(brand_share_m[brand_share_m['bsr_ctgry']=='Bed Frames'])

     bsr_ctgry  year      brand_new  bsr_cnt  year_total_cnt  bsr_share
4   Bed Frames  2021          ZINUS     3104            7079   0.438480
2   Bed Frames  2021         OTHERS     2668            7079   0.376889
1   Bed Frames  2021  AMAZON BASICS     1231            7079   0.173895
0   Bed Frames  2021        ALLEWIE       52            7079   0.007346
3   Bed Frames  2021      YAHEETECH       24            7079   0.003390
10  Bed Frames  2022          ZINUS     3582            7220   0.496122
8   Bed Frames  2022         OTHERS     2291            7220   0.317313
6   Bed Frames  2022  AMAZON BASICS      843            7220   0.116759
9   Bed Frames  2022      YAHEETECH      353            7220   0.048892
5   Bed Frames  2022        ALLEWIE      125            7220   0.017313
7   Bed Frames  2022        LIKIMIO       26            7220   0.003601
18  Bed Frames  2023         OTHERS     3002            7300   0.411233
20  Bed Frames  2023          ZINUS     2084            7300   0

In [160]:
brand_share_m.to_csv('bsr_share_year_0127.csv', index=False)

#### 2. Review rating, review count, Price, 1P3P

In [11]:
df2 = df_bsr.copy()

df2['bsr_date'] = pd.to_datetime(df2['bsr_date'], errors='coerce')
df2['year'] = df2['bsr_date'].dt.year

In [13]:
print(df2)

       parent_asin   bsr_ctgry        asin  rank   bsr_date   brand brand_raw  \
0       B01L3L9WB2  Mattresses  B01L3L9WB2    15 2022-11-09  Others    Others   
1       B0D5LCS7ZW        Beds  B07C2JGHCP    10 2022-11-09  Others    Others   
2       B0B4LPL9HQ        Beds  B0B46NMKLX     1 2022-11-09  Others    Others   
3       B0DKFG8TSB  Bed Frames  B07C2VZ4G5    18 2022-11-09  Others    Others   
4       B0CB5XFRB4        Beds  B09QL28HYV    19 2022-11-09  Others    Others   
...            ...         ...         ...   ...        ...     ...       ...   
108393  B0D8K4N7KM  Mattresses  B0CKYRS739    12 2025-01-27   ZINUS     ZINUS   
108394  B0DHRTBNV1  Bed Frames  B0CSYBPMW4    14 2025-01-27   ZINUS     ZINUS   
108395  B0DYZ28XZV  Mattresses  B0CP1LR1PW     4 2025-01-27   ZINUS     ZINUS   
108396  B0DNSJJBX6  Mattresses  B0CKYV45TS    17 2025-01-27   ZINUS     ZINUS   
108397  B0F198BNN7  Mattresses  B0CSJTKX7K     5 2025-01-27   ZINUS     ZINUS   

       brand_raw_org       

In [23]:
categories = ['Beds', 'Bed Frames', 'Mattresses']

top10_brand_2025 = {
    cat: get_brand_share_by_category(df2, category=cat, years=[2025])
    for cat in categories
}

In [27]:
#print(top10_brand_2025)
top10_brand_2025['Bed Frames']

,brand_raw_org,bsr_cnt,cat_total_cnt,bsr_share,bsr_share_pct
0,ALLEWIE,857,7300,0.117397,11.739726
1,AMAZON BASICS,733,7300,0.100411,10.041096
46,YAHEETECH,491,7300,0.067260,6.726027
48,ZINUS,490,7300,0.067123,6.712329
15,FURNULEM,431,7300,0.059041,5.904110
34,NEW JETO,365,7300,0.050000,5.000000
33,MELLOW,365,7300,0.050000,5.000000
27,LIKIMIO,365,7300,0.050000,5.000000
18,HLIPHA,365,7300,0.050000,5.000000
16,HAFENPO,365,7300,0.050000,5.000000


In [49]:
# Category별로 Top Brand의 data 추출하기
df_2025 = df2[df2['year']==2025].copy()

for cat, df_top10 in top10_brand_2025.items():
    if cat != 'Mattresses': 
        continue

    top_brands = df_top10['brand_raw_org'].unique().tolist()
    
    # ① 해당 category만 필터링
    df_cat = df_2025[df_2025['bsr_ctgry'] == cat].copy()
    if df_cat.empty:
        continue

    # ② 해당 category의 Top10 브랜드만 필터링
    df_cat_top10 = df_cat[df_cat['brand_raw_org'].isin(top_brands)].copy()

    # 동일 시간 중복이면 "마지막"이 남도록 정렬 후 tail(1)
    df_cat_top10 = df_cat_top10.sort_values(['asin', 'bsr_date'])

    #df_cat_top10[df_cat_top10['brand_raw_org']=='ZINUS'].to_csv('test_0113.csv')

    df_latest = (
        df_cat_top10
        .groupby(['bsr_ctgry','asin'], as_index=False)
        .tail(1)
    )
    
    # 1) (category, brand)별 seller_type(1P/2P/3P) ASIN 개수 → wide 형태
    seller_cnt = (
        df_latest
        .pivot_table(
            index=['bsr_ctgry', 'brand_raw_org'],
            columns='seller_type',
            values='asin',
            aggfunc=pd.Series.nunique,
            fill_value=0
        )
        .reset_index()
    )
    #print(seller_cnt)

        # 2) 평균 지표 계산
    metrics = (
        #df_latest
        df_cat_top10
        .groupby(['bsr_ctgry', 'brand_raw_org'], as_index=False)
        .agg(
            avg_rank=('rank', 'mean'),
            avg_rating=('rating', 'mean'),
            avg_retail_price=('retail_price', 'mean'),
            product_cnt=('asin', 'nunique'),   # 전체 ASIN 수(참고용)

            #seller type ratio
            seller_cnt=('seller_type', 'size'), 
            oneP_ratio=('seller_type', lambda x: (x=='1P').mean()),
            twoP_ratio=('seller_type', lambda x: (x=='2P').mean()),
            threeP_ratio=('seller_type', lambda x: (x=='3P').mean())
        )
    )
    #print(metrics)   

    # 3) 결합
    result = (
        metrics
        .merge(seller_cnt, on=['bsr_ctgry', 'brand_raw_org'], how='left')
    )

    print(result) 
    #print(f"\n==== Category: {cat} | unique_asin={df_latest['asin'].nunique()} ====\n\n")    

    bsr_ctgry        brand_raw_org   avg_rank  avg_rating  avg_retail_price  \
0  Mattresses  BEST PRICE MATTRESS  10.747875    4.327616        100.499678   
1  Mattresses             COOL GEL     15.048    4.455600        494.900200   
2  Mattresses              EGOHOME   4.443114    4.571903        119.315378   
3  Mattresses                  FDW   1.879452    4.400000         96.612521   
4  Mattresses               GAESTE    7.12945    4.500324        129.954272   
5  Mattresses                MLILY  14.593002    4.404972        344.252486   
6  Mattresses              MOLBIUS  10.113793    4.335069        156.272222   
7  Mattresses              NOVILLA  10.013834    4.431017        199.643867   
8  Mattresses            ROLANSTAR  14.784689    4.611058         64.869808   
9  Mattresses                ZINUS   7.244076    4.340284        141.891914   

   product_cnt  seller_cnt  oneP_ratio  twoP_ratio  threeP_ratio  1P  2P  3P  
0           12         353    0.915014    0.000000 

#### 3. Brand별 매출 

In [117]:
df3 = df_bsr.copy()

df3['bsr_date'] = pd.to_datetime(df3['bsr_date'], errors='coerce')
df3['year'] = df3['bsr_date'].dt.year

In [119]:
print(df3)

       parent_asin   bsr_ctgry        asin  rank   bsr_date   brand brand_raw  \
0       B084Q4Z5ZJ        Beds  B081HXZ5NK    17 2022-10-23  Others    Others   
1       B0B38RKRB3  Mattresses  B0827DFY3B    19 2022-10-23  Others    Others   
2       B0BSRPX3YF        Beds  B092LNV521    16 2022-10-23  Others    Others   
3       B0CSZLZM7L  Mattresses  B07ZKMYFJZ    17 2022-10-23  Others    Others   
4       B0DKFG8TSB  Bed Frames  B07C2VZ4G5    19 2022-10-23  Others    Others   
...            ...         ...         ...   ...        ...     ...       ...   
108393  B0DHRTBNV1  Bed Frames  B0CSYBPMW4     2 2024-11-12   ZINUS     ZINUS   
108394  B0F193LJW8  Mattresses  B0CSK2VDKG     6 2024-11-12   ZINUS     ZINUS   
108395  B0D8K4N7KM  Mattresses  B0CKYRS739    14 2024-11-12   ZINUS     ZINUS   
108396  B0DDXJLFWD        Beds  B071JGBZYW    17 2024-11-12   ZINUS     ZINUS   
108397  B0DYZ28XZV  Mattresses  B0CP1LR1PW     2 2024-11-12   ZINUS     ZINUS   

       brand_raw_org       

In [121]:
categories = ['Beds', 'Bed Frames', 'Mattresses']

top10_2025 = {
    cat: get_brand_share_by_category(df3, category=cat, years=[2025])
    for cat in categories
}

In [123]:
for cat, df_top in top10_2025.items():
    top10_2025[cat]['brand_raw_org'] = (
        df_top['brand_raw_org']
        .replace({'AMAZON BASICS': 'AMAZONBASICS'})
    )

In [125]:
top10_2025['Beds']

,brand_raw_org,bsr_cnt,cat_total_cnt,bsr_share,bsr_share_pct
59,SHA CERLIN,652,7300,0.089315,8.931507
15,DICTAC,590,7300,0.080822,8.082192
1,ADORNEVE,464,7300,0.063562,6.356164
2,ALLEWIE,389,7300,0.053288,5.328767
3,AMERLIFE,386,7300,0.052877,5.287671
72,VINGLI,383,7300,0.052466,5.246575
47,MELLOW,365,7300,0.050000,5.000000
79,ZINUS,365,7300,0.050000,5.000000
48,MILLIARD,355,7300,0.048630,4.863014
35,JOCOEVOL,345,7300,0.047260,4.726027


In [128]:
df_sales['Brand_up'] = df_sales['Brand'].str.upper().str.strip()
df_sales['Year'] = pd.to_datetime(df_sales['WeekEnding'], errors='coerce').dt.year

In [130]:
print(df_sales)

         RetailerId RetailerName RetailerSku  upc          ModelNumber  \
0                 1   Amazon.com  B09PFVMWKC  NaN                 None   
1                 1   Amazon.com  B082V3R9YM  NaN                 None   
2                 1   Amazon.com  B07YQRL4G5  NaN                 King   
3                 1   Amazon.com  B07DC3JJXN  NaN            AR9016031   
4                 1   Amazon.com  B09BJ4WMMD  NaN             AM-STT-Q   
...             ...          ...         ...  ...                  ...   
2967631           1   Amazon.com  B0DQ885S4N  NaN  14 in king mattress   
2967632           1   Amazon.com  B0DDPQZQYB  NaN          SW-TFC4F-H2   
2967633           1   Amazon.com  B0876Z43S2  NaN           MSHEBT-13Q   
2967634           1   Amazon.com  B0CSFGYRHT  NaN           AOMS10QWP6   
2967635           1   Amazon.com  B0DHXMG94Z  NaN         DYO-14K-BLM2   

                                                     Title        Brand  \
0        14 Inch 4000lbs Heavy Duty 

In [140]:
result_list = []   # ⭐ 결과 누적용 리스트

for cat, df_top10 in top10_2025.items():
    #if cat != 'Beds': 
    #    continue
    #df_top10['brand_raw_org'] = df['brand_raw_org'].replace({'AMAZON BASICS':'AMAZONBASICS'})

    top_brands = df_top10['brand_raw_org'].unique().tolist()
    print(top_brands)

    # ① 해당 category만 필터링
    df_cat = df_sales[df_sales['SubCategory'] == cat].copy()
    if df_cat.empty:
        continue
    
    # ② 해당 category의 Top10 브랜드만 필터링
    df_cat_top10 = df_cat[df_cat['Brand_up'].isin(top_brands)].copy()

    # 2) Brand별 매출 합
    top_brand_sales = (
        df_cat_top10
        .groupby(['SubCategory', 'Year', 'Brand_up'], as_index=False)
        .agg(
            sales=('RetailSales', 'sum'),
            qty=('UnitsSold', 'sum'),
            asins=('RetailerSku', 'nunique')
        )
        .sort_values(   
            by=['SubCategory', 'Year', 'sales'],
            ascending=[True, True, False]
        )
    )

    top_brand_sales['sales'] = top_brand_sales['sales'].map('{:,.0f}'.format)
    result_list.append(top_brand_sales)
    
    print(top_brand_sales)

# ✅ 모든 category 결과 병합
df_result = pd.concat(result_list, ignore_index=True)

    

['SHA CERLIN', 'DICTAC', 'ADORNEVE', 'ALLEWIE', 'AMERLIFE', 'VINGLI', 'MELLOW', 'ZINUS', 'MILLIARD', 'JOCOEVOL']
   SubCategory  Year    Brand_up        sales     qty  asins
8         Beds  2022       ZINUS  146,398,749  741416    475
6         Beds  2022  SHA CERLIN   51,802,193  300110    285
1         Beds  2022     ALLEWIE   47,235,043  249771    268
4         Beds  2022      MELLOW   18,333,672  103560     88
2         Beds  2022    AMERLIFE    4,800,577   22581     43
0         Beds  2022    ADORNEVE    3,594,131   15182     58
3         Beds  2022      DICTAC    1,707,819    7851     63
5         Beds  2022    MILLIARD      138,855    1400      2
7         Beds  2022      VINGLI       47,558     213      7
17        Beds  2023       ZINUS   91,719,519  518072    409
10        Beds  2023     ALLEWIE   68,001,443  386626    421
15        Beds  2023  SHA CERLIN   37,498,166  235705    367
11        Beds  2023    AMERLIFE   25,950,470   93796    113
13        Beds  2023      MELLOW 

In [142]:
df_result.to_csv('sales_asins_year_0127.csv', index=False) 